In [3]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter(action="ignore", category=FutureWarning)

    import os, math
    from pathlib import Path
    import scanpy as sc
    import numpy as np
    import seaborn as sns
    import pandas as pd
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    from scipy import stats
    from scipy.stats import pearsonr, spearmanr
    from scipy.sparse import issparse

    from scipy.stats import wilcoxon, rankdata
    from statsmodels.stats.multitest import multipletests
    from itertools import combinations

    pd.set_option("display.max_rows", 500)
    pd.set_option("display.max_columns", None)

os.chdir(".././")

In [4]:
obs_full = pd.read_csv(
    "5.1_3_obs_all_scores_and_bins.csv", index_col=0, engine="python"
)
obs_full.rename(columns={"sp_score": "sp", "sid_score": "sid"}, inplace=True)
print(obs_full.shape)
obs_full.sample(2)

(1253024, 27)


,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin,cell_type,n_expr_genes,sp,RecSID,sid,ds,fridman,cellage,senmayo,sp_centered_median,sid_centered_median,ds_centered_median,sp_is_sen_median_ct,sid_is_sen_median_ct,ds_is_sen_median_ct,sp_is_sen_median_gse,sid_is_sen_median_gse,ds_is_sen_median_gse,sp_is_sen_median_gsm,sid_is_sen_median_gsm,ds_is_sen_median_gsm
TCTTCGGCATGTAGTC-1-GSM4058914,GSE136831,GSM4058914,66.0,Lung,Macrophage,61.0,Monocytes/macrophages,1263.0,0.000000,SID3,0.852987,1.78324,0.047340,0.062115,0.070039,0.006667,0.047605,0.579014,False,False,False,False,False,False,False,False,False
HCAHeart7905332_CATTCTAGTTGGTGTT-1,Heart_Cell,D6,68.0,Heart_Cell,Mural cell,61.0,Mural cells,2500.0,-0.052672,SID3,0.183087,-0.56950,0.153412,0.045750,0.038796,-0.217485,-0.418698,0.684740,False,False,True,False,False,True,False,False,True


In [5]:
obs_dw = pd.read_csv("5.3_dw_obs_all_scores_and_bins.csv", index_col=0, engine="python")
obs_dw.rename(columns={"sp_score": "sp", "sid_score": "sid"}, inplace=True)
print(obs_dw.shape)
obs_dw.sample(2)

(1253024, 27)


,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin,cell_type,n_expr_genes,sp,RecSID,sid,ds,fridman,cellage,senmayo,sp_centered_median,sid_centered_median,ds_centered_median,sp_is_sen_median_ct,sid_is_sen_median_ct,ds_is_sen_median_ct,sp_is_sen_median_gse,sid_is_sen_median_gse,ds_is_sen_median_gse,sp_is_sen_median_gsm,sid_is_sen_median_gsm,ds_is_sen_median_gsm
TGGGCTGGTTTACCTT-1-GSM5364335,GSE176415,GSM5364335,23.0,Skin,Fibroblast,21.0,Fibroblasts,3048.0,48.670025,SID3,0.842969,1.011407,0.134573,0.050880,0.061178,6.983496,0.513614,0.084023,False,True,False,False,True,False,False,True,False
H0037_Apex_CCGTAGGGTTACAGCT,Heart_Nuclei,H4,58.0,Heart_Nuclei,Ventricular Cardiomyocyte,51.0,Ventricular cardiomyocytes,2999.0,5.078528,SID3,0.936991,-0.019876,0.020285,0.023945,0.003415,3.786668,0.140600,1.477613,True,False,True,True,False,True,True,True,True


In [6]:
sen_sigs = ["fridman", "senmayo", "cellage"]
methods = ["sp", "ds", "sid"]

In [7]:
"""assign Sc/NSc groups (NSc = ~Sc)"""

cols_to_drop = obs_full.filter(
    regex="_is_sen_median_ct$|_is_sen_median_gsm$|_centered_median$"
).columns

for df in obs_full, obs_dw:
    df.drop(columns=cols_to_drop, inplace=True)
    for m in methods:
        # Создаем маски для фильтрации
        mask_sc = df[f"{m}_is_sen_median_gse"] == True
        mask_nsc = df[f"{m}_is_sen_median_gse"] == False

        # Сначала инициализируем колонку значениями pd.NA
        df[f"{m}_sen_group"] = pd.NA

        # Заполняем группы на основе масок
        df.loc[mask_sc, f"{m}_sen_group"] = "Sc"
        df.loc[mask_nsc, f"{m}_sen_group"] = "NSc"

In [8]:
obs_full.head(2)

,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin,cell_type,n_expr_genes,sp,RecSID,sid,ds,fridman,cellage,senmayo,sp_is_sen_median_gse,sid_is_sen_median_gse,ds_is_sen_median_gse,sp_sen_group,ds_sen_group,sid_sen_group
AAACCTGAGGCCGAAT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21.0,Fibroblasts,950.0,3.951097,SID3,0.773698,0.156639,0.222860,0.048563,0.124413,False,True,False,NSc,NSc,Sc
AAACCTGAGTTAAGTG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD8T,21.0,CD8 T cells,1247.0,0.114196,SID3,0.607294,0.125290,0.096653,0.047017,0.039243,False,False,False,NSc,NSc,NSc


In [9]:
uni_cols = ["GSE_id", "GSM_id", "Tissue_global", "cell_type", "cell_type_final", "Age"]

# 1. Заранее переводим uni_cols в категории для ускорения будущей группировки
for col in uni_cols:
    if obs_full[col].dtype == "object":
        obs_full[col] = obs_full[col].astype("category")

# 2. Мерджим по индексам клеток
# Из obs_dw мы временно исключаем uni_cols, чтобы они не дублировались
obs_all = pd.merge(
    obs_full,
    obs_dw.drop(columns=uni_cols, errors="ignore"),
    left_index=True,
    right_index=True,
    suffixes=("_full", "_dw"),
)


In [10]:
print(obs_all.shape)
obs_all.head(2)

(1253024, 36)


,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin_full,cell_type,n_expr_genes_full,sp_full,RecSID_full,sid_full,ds_full,fridman_full,cellage_full,senmayo_full,sp_is_sen_median_gse_full,sid_is_sen_median_gse_full,ds_is_sen_median_gse_full,sp_sen_group_full,ds_sen_group_full,sid_sen_group_full,Age_bin_dw,n_expr_genes_dw,sp_dw,RecSID_dw,sid_dw,ds_dw,fridman_dw,cellage_dw,senmayo_dw,sp_is_sen_median_gse_dw,sid_is_sen_median_gse_dw,ds_is_sen_median_gse_dw,sp_sen_group_dw,ds_sen_group_dw,sid_sen_group_dw
AAACCTGAGGCCGAAT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21.0,Fibroblasts,950.0,3.951097,SID3,0.773698,0.156639,0.222860,0.048563,0.124413,False,True,False,NSc,NSc,Sc,21.0,950.0,14.166038,SID3,0.773698,0.145985,0.195521,0.046323,0.124120,False,True,False,NSc,NSc,Sc
AAACCTGAGTTAAGTG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD8T,21.0,CD8 T cells,1247.0,0.114196,SID3,0.607294,0.125290,0.096653,0.047017,0.039243,False,False,False,NSc,NSc,NSc,21.0,1247.0,2.127291,SID3,0.607294,0.124610,0.080760,0.051311,0.040026,False,False,True,NSc,Sc,NSc


In [11]:
group_cols = ["GSM_id", "Tissue_global", "cell_type"]


all_results = []


# Вспомогательная функция, которая быстро считает метрики для одной группы
# Pandas передает сюда подтаблицу сразу целиком
def calc_metrics_for_group(df_g, m):
    x = df_g[f"{m}_full"].to_numpy()
    y = df_g[f"{m}_dw"].to_numpy()

    # Быстрые проверки std и уникальных значений
    p_r, p_p = pearsonr(x, y) if np.std(x) > 0 and np.std(y) > 0 else (np.nan, np.nan)
    s_r, s_p = (
        spearmanr(x, y)
        if len(np.unique(x)) > 1 and len(np.unique(y)) > 1
        else (np.nan, np.nan)
    )

    return pd.Series(
        {
            "n_cells": len(df_g),
            "pearson_r": p_r,
            "pearson_p": p_p,
            "spearman_rho": s_r,
            "spearman_p": s_p,
        }
    )


for m in methods:
    # 1. Выделяем метаданные и нужный метод
    obs_sub = obs_all[group_cols + [f"{m}_dw", f"{m}_full"]].dropna(
        subset=[f"{m}_dw", f"{m}_full"]
    )

    # Оптимизация: переводим категориальные колонки в тип category, если они объекты.
    # Это многократно ускоряет внутренний groupby в Pandas!
    for col in group_cols:
        if obs_sub[col].dtype == "object":
            obs_sub[col] = mergedobs_sub_df[col].astype("category")

    # 3. Векторизованный groupby через .apply() вместо цикла Python
    # group_keys=False предотвращает дублирование индексов
    df_method = (
        obs_sub.groupby(group_cols, observed=True, group_keys=False)
        .apply(calc_metrics_for_group, m=m, include_groups=False)
        .reset_index()
    )

    # Если для метода нет валидных строк, пропускаем
    if df_method.empty or "n_cells" not in df_method.columns:
        continue

    # Добавляем имя метода
    df_method["method"] = m

    # 4. Локальная FDR коррекция внутри метода
    df_method["pearson_p_adj"] = np.nan
    df_method["spearman_p_adj"] = np.nan

    valid_p = df_method["pearson_p"].notna()
    if valid_p.any():
        df_method.loc[valid_p, "pearson_p_adj"] = multipletests(
            df_method.loc[valid_p, "pearson_p"], method="fdr_bh"
        )[1]

    valid_s = df_method["spearman_p"].notna()
    if valid_s.any():
        df_method.loc[valid_s, "spearman_p_adj"] = multipletests(
            df_method.loc[valid_s, "spearman_p"], method="fdr_bh"
        )[1]

    all_results.append(df_method)

# Собираем всё вместе
obs_corr = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


In [12]:
"""агрегация корреляций - насколько стабильно 
методы согласуются между donor × tissue × cell type"""

corr_summary = (
    obs_corr.groupby("method", observed=True)
    .agg(
        n_groups=("spearman_rho", "count"),
        median_spearman=("spearman_rho", "median"),
        q25_spearman=("spearman_rho", lambda x: x.quantile(0.25)),
        q75_spearman=("spearman_rho", lambda x: x.quantile(0.75)),
        median_pearson=("pearson_r", "median"),
        q25_pearson=("pearson_r", lambda x: x.quantile(0.25)),
        q75_pearson=("pearson_r", lambda x: x.quantile(0.75)),
        frac_positive_spearman=("spearman_rho", lambda x: (x > 0).mean()),
    )
    .reset_index()
)

corr_summary

,method,n_groups,median_spearman,q25_spearman,q75_spearman,median_pearson,q25_pearson,q75_pearson,frac_positive_spearman
0,ds,1116,0.669054,0.029955,0.922560,0.660786,0.012331,0.933760,0.762545
1,sid,1126,0.934495,0.846044,0.981890,0.929638,0.840710,0.979754,1.000000
2,sp,697,0.418417,0.015766,0.745817,0.648088,0.356101,0.869756,0.753228


In [13]:
obs_corr.head(2)

,GSM_id,Tissue_global,cell_type,n_cells,pearson_r,pearson_p,spearman_rho,spearman_p,method,pearson_p_adj,spearman_p_adj
0,A61,Heart_Nuclei,Atrial cardiomyocytes,3200.0,0.832474,0.000000e+00,0.83236,0.000000e+00,sp,0.000000e+00,0.000000e+00
1,A61,Heart_Nuclei,CD4 T cells,332.0,0.958413,8.986704e-182,0.52759,3.471729e-25,sp,3.796202e-181,7.034289e-25


#### binary concordance: Sc/NSc

In [18]:
obs_all.columns

Index(['GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final',
       'Age_bin_full', 'cell_type', 'n_expr_genes_full', 'sp_full',
       'RecSID_full', 'sid_full', 'ds_full', 'fridman_full', 'cellage_full',
       'senmayo_full', 'sp_is_sen_median_gse_full',
       'sid_is_sen_median_gse_full', 'ds_is_sen_median_gse_full',
       'sp_sen_group_full', 'ds_sen_group_full', 'sid_sen_group_full',
       'Age_bin_dw', 'n_expr_genes_dw', 'sp_dw', 'RecSID_dw', 'sid_dw',
       'ds_dw', 'fridman_dw', 'cellage_dw', 'senmayo_dw',
       'sp_is_sen_median_gse_dw', 'sid_is_sen_median_gse_dw',
       'ds_is_sen_median_gse_dw', 'sp_sen_group_dw', 'ds_sen_group_dw',
       'sid_sen_group_dw'],
      dtype='object')

In [14]:
from sklearn.metrics import cohen_kappa_score


# Вспомогательная векторизованная функция расчетов для одной группы
def calc_binary_metrics(df_g, col_full, col_dw):
    # Достаем массивы меток
    a = df_g[col_full].to_numpy()
    b = df_g[col_dw].to_numpy()

    # Переводим в бинарный вид (Sc = 1, NSc = 0)
    a_bin = (a == "Sc").astype(int)
    b_bin = (b == "Sc").astype(int)

    # Базовая матрица ошибок
    TP = int(np.sum((a_bin == 1) & (b_bin == 1)))
    FP = int(np.sum((a_bin == 1) & (b_bin == 0)))
    FN = int(np.sum((a_bin == 0) & (b_bin == 1)))
    TN = int(np.sum((a_bin == 0) & (b_bin == 0)))

    # Метрики пересечения и согласия
    union = TP + FP + FN
    jaccard = TP / union if union > 0 else np.nan

    # Полнота и точность (доли совпадений)
    overlap_full_given_dw = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    overlap_dw_given_full = TP / (TP + FP) if (TP + FP) > 0 else np.nan

    agreement = (TP + TN) / len(df_g)

    # Каппа Коэна
    if len(np.unique(a_bin)) > 1 and len(np.unique(b_bin)) > 1:
        kappa = cohen_kappa_score(a_bin, b_bin)
    else:
        kappa = np.nan

    return pd.Series(
        {
            "n_cells": len(df_g),
            "n_Sc_full": int(a_bin.sum()),
            "n_Sc_dw": int(b_bin.sum()),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "TN": TN,
            "sc_fraction_full": float(a_bin.mean()),
            "sc_fraction_dw": float(b_bin.mean()),
            "jaccard": jaccard,
            "overlap_full_given_dw": overlap_full_given_dw,
            "overlap_dw_given_full": overlap_dw_given_full,
            "agreement": agreement,
            "cohen_kappa": kappa,
        }
    )


all_results_binary = []

for m in methods:
    col_full = f"{m}_sen_group_full"
    col_dw = f"{m}_sen_group_dw"

    # Оставляем только строки, где есть обе метки для текущего метода
    df_method_valid = obs_all.dropna(subset=[col_full, col_dw])

    if df_method_valid.empty:
        continue

    # Быстрый векторизованный расчет по группам клеток
    df_res_method = (
        df_method_valid.groupby(group_cols, observed=True, group_keys=False)
        .apply(
            calc_binary_metrics,
            col_full=col_full,
            col_dw=col_dw,
            include_groups=False,
        )
        .reset_index()
    )

    if df_res_method.empty or "n_cells" not in df_res_method.columns:
        continue

    # Добавляем имя метода в результат
    df_res_method.insert(0, "method", m)

    all_results_binary.append(df_res_method)

# Объединяем все методы в итоговый датафрейм
binary_df = (
    pd.concat(all_results_binary, ignore_index=True)
    if all_results_binary
    else pd.DataFrame()
)


In [15]:
binary_df

,method,GSM_id,Tissue_global,cell_type,n_cells,n_Sc_full,n_Sc_dw,TP,FP,FN,TN,sc_fraction_full,sc_fraction_dw,jaccard,overlap_full_given_dw,overlap_dw_given_full,agreement,cohen_kappa
0,sp,A61,Heart_Nuclei,Atrial cardiomyocytes,3200.0,1099.0,1009.0,777.0,322.0,232.0,1869.0,0.343438,0.315312,0.583772,0.770069,0.707006,0.826875,0.608465
1,sp,A61,Heart_Nuclei,CD4 T cells,332.0,111.0,143.0,90.0,21.0,53.0,168.0,0.334337,0.430723,0.548780,0.629371,0.810811,0.777108,0.532768
2,sp,A61,Heart_Nuclei,Endothelial cells,2508.0,904.0,855.0,768.0,136.0,87.0,1517.0,0.360447,0.340909,0.774975,0.898246,0.849558,0.911085,0.804837
3,sp,A61,Heart_Nuclei,Fibroblasts,9509.0,4546.0,2922.0,2535.0,2011.0,387.0,4576.0,0.478073,0.307288,0.513886,0.867556,0.557633,0.747818,0.486964
4,sp,A61,Heart_Nuclei,Monocytes/macrophages,1903.0,565.0,374.0,306.0,259.0,68.0,1270.0,0.296900,0.196532,0.483412,0.818182,0.541593,0.828166,0.543882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2934,sid,HRS119002,Skin,Undiff. keratinocytes,1260.0,477.0,448.0,385.0,92.0,63.0,720.0,0.378571,0.355556,0.712963,0.859375,0.807128,0.876984,0.735405
2935,sid,HRS119003,Skin,Diff. keratinocytes,3618.0,996.0,956.0,883.0,113.0,73.0,2549.0,0.275290,0.264234,0.826006,0.923640,0.886546,0.948590,0.869533
2936,sid,HRS119003,Skin,Undiff. keratinocytes,1534.0,566.0,546.0,529.0,37.0,17.0,951.0,0.368970,0.355932,0.907376,0.968864,0.934629,0.964798,0.923845
2937,sid,HRS119004,Skin,Diff. keratinocytes,3396.0,822.0,809.0,779.0,43.0,30.0,2544.0,0.242049,0.238221,0.914319,0.962917,0.947689,0.978504,0.941099


In [16]:
"""aggreration"""
binary_summary = (
    binary_df.groupby(["method"], observed=True)
    .agg(
        n_groups=("jaccard", "count"),
        median_jaccard=("jaccard", "median"),
        q25_jaccard=("jaccard", lambda x: x.quantile(0.25)),
        q75_jaccard=("jaccard", lambda x: x.quantile(0.75)),
        median_overlap_full_given_dw=("overlap_full_given_dw", "median"),
        median_overlap_dw_given_full=("overlap_dw_given_full", "median"),
        median_agreement=("agreement", "median"),
        median_kappa=("cohen_kappa", "median"),
        q25_kappa=("cohen_kappa", lambda x: x.quantile(0.25)),
        q75_kappa=("cohen_kappa", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

binary_summary

,method,n_groups,median_jaccard,q25_jaccard,q75_jaccard,median_overlap_full_given_dw,median_overlap_dw_given_full,median_agreement,median_kappa,q25_kappa,q75_kappa
0,ds,1111,0.269461,0.075995,0.589836,0.478261,0.549854,0.883554,0.341131,0.050632,0.702396
1,sid,1034,0.769231,0.554421,0.905856,0.887500,0.899225,0.958981,0.834134,0.665075,0.936305
2,sp,695,0.285714,0.153203,0.488724,0.555556,0.500000,0.870968,0.352731,0.163567,0.594631


In [42]:
from matplotlib.ticker import FormatStrFormatter, LinearLocator

# Фиксированный порядок методов по оси Y
methods_order = ["sp", "ds", "sid"]

# 1. Подготовка данных BINARY (Jaccard, Kappa)
binary_df["group"] = (
    binary_df["Tissue_global"].astype(str) + " | " + binary_df["cell_type"].astype(str)
)

# Агрегируем по методу и биологической группе клеток (медиана по всем GSM_id)
binary_summary = (
    binary_df.groupby(["method", "group"], observed=True)[["jaccard", "cohen_kappa"]]
    .median()
    .reset_index()
)

# 2. Подготовка данных CORRELATION (Spearman)
corr_df = obs_corr.copy()
corr_df["group"] = (
    corr_df["Tissue_global"].astype(str) + " | " + corr_df["cell_type"].astype(str)
)

# Агрегируем Spearman (медиана по всем GSM_id)
spearman_summary = (
    corr_df.groupby(["method", "group"], observed=True)[
        ["spearman_rho", "spearman_p_adj"]
    ]
    .median()
    .reset_index()
)

# 3. Единый порядок по оси X (алфавитный по тканям и типам клеток)
group_order = (
    pd.concat(
        [
            corr_df[["Tissue_global", "cell_type", "group"]],
            binary_df[["Tissue_global", "cell_type", "group"]],
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values(["Tissue_global", "cell_type"])["group"]
    .tolist()
)

# 4. Разворачиваем (Pivot) сводные таблицы в матрицы для Heatmap
spearman_matrix = spearman_summary.pivot(
    index="method", columns="group", values="spearman_rho"
).reindex(index=methods_order, columns=group_order)

spearman_p_matrix = spearman_summary.pivot(
    index="method", columns="group", values="spearman_p_adj"
).reindex(index=methods_order, columns=group_order)

jaccard_matrix = binary_summary.pivot(
    index="method", columns="group", values="jaccard"
).reindex(index=methods_order, columns=group_order)

kappa_matrix = binary_summary.pivot(
    index="method", columns="group", values="cohen_kappa"
).reindex(index=methods_order, columns=group_order)


# 5. Функция-помощник для разметки пропусков (NaN)
def add_missing_marks(ax, data):
    for row_idx in range(data.shape[0]):
        for col_idx in range(data.shape[1]):
            value = data.iloc[row_idx, col_idx]
            if pd.isna(value):
                ax.text(
                    col_idx + 0.5,
                    row_idx + 0.5,
                    "–",
                    ha="center",
                    va="center",
                    fontsize=35,
                    fontweight="normal",
                )


# 6. Настройка динамической ширины фигуры под количество групп клеток
fig_width = max(12, len(group_order) * 0.9)
fig = plt.figure(figsize=(fig_width, 18), dpi=300)

gs = fig.add_gridspec(
    nrows=3,
    ncols=2,
    width_ratios=[0.01, 1],
    height_ratios=[1, 1, 1],
    wspace=0.1,
    hspace=0.12,
)

# Оси для колорбаров (слева)
cbar_axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[2, 0]),
]

# Оси для самих хитмапов (справа)
ax0 = fig.add_subplot(gs[0, 1])
ax1 = fig.add_subplot(gs[1, 1], sharex=ax0)
ax2 = fig.add_subplot(gs[2, 1], sharex=ax0)
axes = [ax0, ax1, ax2]

# --- 7. ПАНЕЛЬ 1: SPEARMAN ---
ax = axes[0]
sns.heatmap(
    spearman_matrix,
    annot=False,
    vmin=0,
    vmax=1,  # Для само-корреляции full/dw логично смотреть диапазон от 0 до 1
    cmap="BrBG_r",
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    cbar_kws={"shrink": 0.8, "aspect": 10, "pad": 0.01},
    ax=ax,
    cbar_ax=cbar_axes[0],
)
add_missing_marks(ax, spearman_matrix)

# Нанесение звездочек значимости (FDR-adjusted p-value < 0.05)
for row_idx in range(spearman_matrix.shape[0]):
    for col_idx in range(spearman_matrix.shape[1]):
        val = spearman_matrix.iloc[row_idx, col_idx]
        p_adj = spearman_p_matrix.iloc[row_idx, col_idx]
        if pd.notna(val) and pd.notna(p_adj) and p_adj < 0.05:
            ax.text(
                col_idx + 0.5,
                row_idx + 0.5,
                "∗",
                ha="center",
                va="center",
                fontsize=40,
                fontweight="bold",
            )

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=32, length=3)
cbar.set_label("Spearman ρ", fontsize=50, fontweight="semibold", labelpad=12)


# --- 8. ПАНЕЛЬ 2: JACCARD ---
ax = axes[1]
sns.heatmap(
    jaccard_matrix,
    annot=False,
    vmin=0,
    vmax=1,
    cmap="YlGnBu",
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    cbar_kws={"shrink": 0.8, "aspect": 10, "pad": 0.025},
    ax=ax,
    cbar_ax=cbar_axes[1],
)
add_missing_marks(ax, jaccard_matrix)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=32, length=3)
cbar.set_label("Jaccard", fontsize=50, fontweight="semibold", labelpad=12)


# --- 9. ПАНЕЛЬ 3: COHEN'S KAPPA ---
ax = axes[2]
sns.heatmap(
    kappa_matrix,
    annot=False,
    vmin=0,
    vmax=1,
    cmap="PuRd",
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    cbar_kws={"shrink": 0.8, "aspect": 10, "pad": 0.025},
    ax=ax,
    cbar_ax=cbar_axes[2],
)
add_missing_marks(ax, kappa_matrix)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=32, length=3)
cbar.set_label("Cohen's κ", fontsize=50, fontweight="semibold", labelpad=12)


# --- 10. ФОРМАТИРОВАНИЕ ОСЕЙ И ШРИФТОВ ---
# Отключаем деления X для верхней и средней панели
for ax in axes[:2]:
    ax.tick_params(
        axis="x",
        which="both",
        bottom=False,
        top=False,
        labelbottom=False,
        labeltop=False,
    )
    ax.set_xlabel("")

# Настраиваем только нижнюю ось X
axes[2].set_xticklabels(
    group_order, fontsize=50, fontweight="semibold", rotation=90, ha="center"
)
axes[2].tick_params(axis="x", which="both", bottom=True, top=False, length=10, width=3)
axes[2].set_xlabel("Tissue | cell type", fontsize=68, fontweight="normal", labelpad=15)

# Оформление оси Y (названия методов справа, как в исходном коде)
for ax in axes:
    ax.tick_params(
        axis="y",
        left=False,
        right=True,
        labelleft=False,
        labelright=True,
        length=0,
        width=0,
        pad=8,
    )
    for label in ax.get_yticklabels():
        label.set_fontsize(57)
        label.set_fontweight("semibold")
        label.set_rotation(0)
    ax.set_ylabel("")

    # Перемещаем подписи колорбаров влево
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_ticks_position("right")
    cbar.ax.yaxis.set_label_position("left")

plt.subplots_adjust(left=0.12, right=0.92, bottom=0.20, top=0.94, hspace=0.01)
plt.show()


In [32]:
# Направления условного перекрытия
directions = ["overlap_full_given_dw", "overlap_dw_given_full"]

# Задаем жесткую структуру строк графика: (метод, метрика)
column_structure = []
for m in methods:
    for d in directions:
        column_structure.append((m, d))

# Базовый шрифт для масштабирования элементов
base_size = 32

# 1. Формируем группу Tissue | cell type
binary_df["group"] = (
    binary_df["Tissue_global"].astype(str) + " | " + binary_df["cell_type"].astype(str)
)

# 2. Определяем порядок по оси X (по тканям и типам клеток)
group_info = (
    binary_df[["Tissue_global", "cell_type", "group"]]
    .drop_duplicates()
    .sort_values(["Tissue_global", "cell_type"])
)
group_order = group_info["group"].tolist()

# 3. Агрегируем условные перекрытия по группам (медиана по всем GSM_id)
summary = (
    binary_df.groupby(["Tissue_global", "cell_type", "method"], observed=True)[
        directions
    ]
    .median()
    .reset_index()
)

# 4. Создаем пустую матрицу под новую структуру
conditional_matrix = pd.DataFrame(
    np.nan,
    index=group_order,
    columns=pd.MultiIndex.from_tuples(column_structure, names=["Method", "Direction"]),
)

# 5. Заполняем матрицу
for _, row in summary.iterrows():
    group = str(row["Tissue_global"]) + " | " + str(row["cell_type"])
    m = row["method"]

    if (m, "overlap_full_given_dw") in column_structure:
        conditional_matrix.loc[group, (m, "overlap_full_given_dw")] = row[
            "overlap_full_given_dw"
        ]
    if (m, "overlap_dw_given_full") in column_structure:
        conditional_matrix.loc[group, (m, "overlap_dw_given_full")] = row[
            "overlap_dw_given_full"
        ]

# Транспонируем, чтобы группы клеток ушли на ось X, а метрики на ось Y
conditional_matrix_t = conditional_matrix.T

# 6. Нормализация цвета
conditional_vmin = np.nanmin(conditional_matrix_t.values)
conditional_vmax = np.nanmax(conditional_matrix_t.values)

# 7. Создание фигуры
fig_width = max(12, len(group_order) * 0.9)  # Чуть шире под вертикальные подписи
fig, ax = plt.subplots(figsize=(fig_width, 9), dpi=300)

# 8. Отрисовка хитмапа
sns.heatmap(
    conditional_matrix_t,
    ax=ax,
    cmap="GnBu",
    vmin=conditional_vmin,
    vmax=conditional_vmax,
    annot=False,
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    cbar_kws={"shrink": 1.4, "aspect": 16, "pad": 0.118},
)

# Убираем дефолтные подписи осей
ax.set_xlabel("Tissue | cell type", fontsize=base_size + 13, labelpad=15)
ax.set_ylabel("")

# 9. Настройка оси X (Вертикальные подписи по центру засечки)
ax.tick_params(
    axis="x",
    which="both",
    bottom=True,
    top=False,
    labelbottom=True,
    labeltop=False,
    length=10,
    width=3,
)

ax.set_xticklabels(
    group_order,
    fontsize=base_size + 3,
    fontweight="semibold",
    rotation=90,
    ha="center",  # Выравнивание строго по центру засечки
)

# 10. Прячем дефолтные подписи оси Y
ax.tick_params(
    axis="y",
    left=False,
    right=False,
    labelleft=False,
    labelright=False,
)

# 11. Подписи оцениваемого метода (Method) слева
# Каждому методу теперь принадлежит ровно по 2 строки в матрице
method_groups = {"sp": (0, 2), "ds": (2, 4), "sid": (4, 6)}

for method_name, (start, end) in method_groups.items():
    center = (start + end) / 2
    ax.text(
        -0.38,
        center,
        method_name,
        ha="right",
        va="center",
        fontsize=base_size + 9,
        fontweight="semibold",
        clip_on=False,
    )

# Общий заголовок для методов слева
ax.text(
    -2.5,
    conditional_matrix_t.shape[0] / 2,
    "Method",
    ha="right",
    va="center",
    fontsize=base_size + 13,
    fontweight="normal",
    clip_on=False,
    rotation=90,
)

# 12. Подписи направления сравнения (Direction) справа
# Превращаем технические названия колонок в читаемые лейблы
direction_labels = {
    "overlap_full_given_dw": "Full | Dw",
    "overlap_dw_given_full": "Dw | Full",
}

for row_idx, (_, direction) in enumerate(column_structure):
    label_text = direction_labels.get(direction, direction)
    ax.text(
        len(group_order) + 0.28,
        row_idx + 0.5,
        label_text,
        ha="left",
        va="center",
        fontsize=base_size + 5,
        fontweight="semibold",
        clip_on=False,
    )

# Общий заголовок для направлений справа
ax.text(
    len(group_order) + 7.5,
    conditional_matrix_t.shape[0] / 2,
    "       Comparison Direction",
    ha="center",
    va="center",
    fontsize=base_size + 13,
    fontweight="normal",
    clip_on=False,
    rotation=-90,
)

# 13. Горизонтальные разделители между блоками методов
for y in [2, 4]:
    ax.axhline(
        y,
        color="black",
        linewidth=1.5,
        zorder=10,
    )

# 14. Вертикальные разделители границ тканей (Tissue_global)
tissue_boundaries = (
    binary_df[["Tissue_global", "cell_type"]]
    .drop_duplicates()
    .sort_values(["Tissue_global", "cell_type"])
    .groupby("Tissue_global", observed=True)
    .size()
    .cumsum()
    .iloc[:-1]
)
for boundary in tissue_boundaries:
    ax.axvline(
        boundary,
        color="black",
        linewidth=1.2,
        zorder=10,
    )

# 15. Маркировка отсутствующих значений (NaN)
for row_idx in range(conditional_matrix_t.shape[0]):
    for col_idx in range(conditional_matrix_t.shape[1]):
        value = conditional_matrix_t.iloc[row_idx, col_idx]
        if pd.isna(value):
            ax.text(
                col_idx + 0.5,
                row_idx + 0.5,
                "–",
                ha="center",
                va="center",
                fontsize=base_size,
                fontweight="normal",
            )

# 16. Настройка цветовой шкалы (Colorbar)
cbar = ax.collections[0].colorbar
cbar.ax.yaxis.set_ticks_position("left")
cbar.ax.yaxis.set_label_position("right")

cbar.ax.tick_params(
    axis="y",
    left=True,
    right=False,
    labelleft=True,
    labelright=False,
    length=3,
    width=0.9,
    labelsize=base_size + 3,
)

cbar.set_label(
    "Conditional Sc overlap",
    fontsize=base_size + 7,
    fontweight="semibold",
    rotation=270,
    labelpad=30,
)

# 17. Корректировка отступов лейаута под вертикальный текст
plt.subplots_adjust(
    left=0.15,
    right=0.82,
    bottom=0.35,  # Увеличено место снизу под длинные вертикальные подписи типов клеток
    top=0.90,
)

plt.show()


In [34]:
obs_dw[:10]

,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin,cell_type,n_expr_genes,sp,RecSID,sid,ds,fridman,cellage,senmayo,sp_is_sen_median_gse,sid_is_sen_median_gse,ds_is_sen_median_gse,sp_sen_group,ds_sen_group,sid_sen_group
AAACCTGAGGCCGAAT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21.0,Fibroblasts,950.0,14.166038,SID3,0.773698,0.145985,0.195521,0.046323,0.124120,False,True,False,NSc,NSc,Sc
AAACCTGAGTTAAGTG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD8T,21.0,CD8 T cells,1247.0,2.127291,SID3,0.607294,0.124610,0.080760,0.051311,0.040026,False,False,True,NSc,Sc,NSc
AAACCTGGTCTTCTCG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,VE,21.0,Vein endothelium,724.0,1.607762,SID3,0.877589,0.482569,0.073341,0.071276,0.054778,False,False,False,NSc,NSc,NSc
AAACCTGGTGCAACTT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,DC,21.0,Dendritic cells,1888.0,11.074797,SID3,0.754094,0.569158,0.063165,0.055712,0.055372,False,False,False,NSc,NSc,NSc
AAACCTGTCAGCGACC-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21.0,CD4 T cells,721.0,1.106343,SID3,0.518898,-0.410551,0.033974,0.057780,0.069256,False,False,False,NSc,NSc,NSc
AAACCTGTCATGTGGT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Pericyte,21.0,Pericytes,1183.0,2.276621,SID3,0.799043,0.918122,0.107464,0.053329,0.042529,False,True,False,NSc,NSc,Sc
AAACCTGTCCATTCTA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21.0,CD4 T cells,1387.0,1.491130,SID3,0.745804,1.234035,0.096234,0.060751,0.036938,False,False,False,NSc,NSc,NSc
AAACCTGTCCGGCACA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21.0,CD4 T cells,705.0,0.852130,SID3,0.515885,0.778054,0.055191,0.028279,0.035191,False,False,False,NSc,NSc,NSc
AAACCTGTCGGAGGTA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21.0,Fibroblasts,1048.0,12.417925,SID3,0.453299,1.525131,0.142060,0.035706,0.099105,False,False,False,NSc,NSc,NSc
AAACCTGTCGGATGGA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,DC,21.0,Dendritic cells,1211.0,4.183546,SID3,0.397722,0.411644,0.087449,0.055605,0.084138,False,False,False,NSc,NSc,NSc


In [36]:
obs_full.loc[obs_dw.index[:10], :]

,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin,cell_type,n_expr_genes,sp,RecSID,sid,ds,fridman,cellage,senmayo,sp_is_sen_median_gse,sid_is_sen_median_gse,ds_is_sen_median_gse,sp_sen_group,ds_sen_group,sid_sen_group
AAACCTGAGGCCGAAT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21.0,Fibroblasts,950.0,3.951097,SID3,0.773698,0.156639,0.222860,0.048563,0.124413,False,True,False,NSc,NSc,Sc
AAACCTGAGTTAAGTG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD8T,21.0,CD8 T cells,1247.0,0.114196,SID3,0.607294,0.125290,0.096653,0.047017,0.039243,False,False,False,NSc,NSc,NSc
AAACCTGGTCTTCTCG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,VE,21.0,Vein endothelium,724.0,0.151354,SID3,0.877589,0.529626,0.070124,0.068919,0.055252,False,False,False,NSc,NSc,NSc
AAACCTGGTGCAACTT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,DC,21.0,Dendritic cells,1888.0,1.127572,SID3,0.754094,3.609708,0.071895,0.055063,0.062890,False,False,False,NSc,NSc,NSc
AAACCTGTCAGCGACC-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21.0,CD4 T cells,721.0,0.178643,SID3,0.518898,-0.213985,0.029474,0.056690,0.077574,False,False,False,NSc,NSc,NSc
AAACCTGTCATGTGGT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Pericyte,21.0,Pericytes,1183.0,0.279768,SID3,0.799043,-0.088717,0.116478,0.050916,0.049669,False,True,False,NSc,NSc,Sc
AAACCTGTCCATTCTA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21.0,CD4 T cells,1387.0,0.294915,SID3,0.745804,1.379661,0.103910,0.050480,0.044404,True,False,False,Sc,NSc,NSc
AAACCTGTCCGGCACA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21.0,CD4 T cells,705.0,0.012371,SID3,0.515885,0.751684,0.061313,0.026961,0.034684,False,False,False,NSc,NSc,NSc
AAACCTGTCGGAGGTA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21.0,Fibroblasts,1048.0,1.543067,SID3,0.453299,1.602780,0.143046,0.038246,0.097591,False,False,False,NSc,NSc,NSc
AAACCTGTCGGATGGA-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,DC,21.0,Dendritic cells,1211.0,1.124219,SID3,0.397722,2.678384,0.087720,0.050738,0.094211,False,False,False,NSc,NSc,NSc
